# 01 · Zero- and Few-Shot Prompting of Turkish LLMs on TrGLUE

Four 7–9B instruction models are prompted with **0, 3 and 5 examples** on 200 test sentences each of **SST-2** (sentiment) and **CoLA** (grammatical acceptability). Model outputs are parsed into labels and scored.

| Short name | Model |
|---|---|
| turkcell | `TURKCELL/Turkcell-LLM-7b-v1` |
| cosmos | `ytu-ce-cosmos/Turkish-Llama-8b-DPO-v0.1` |
| koc | `KOCDIGITAL/Kocdigital-LLM-8b-v0.1` |
| gemma | `google/gemma-2-9b-it` |

The notebook has two rounds:
1. **Baseline prompt** on the raw (imbalanced) test split
2. **Improved prompt + stricter output parsing + class-balanced data**

> Runs were executed on a cloud GPU (paths such as `/content` and `/workspace` come from that environment).

In [1]:
import torch
import random
import time
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.manifold import TSNE
import seaborn as sns
import os
import gc
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from huggingface_hub import snapshot_download
from accelerate import init_empty_weights, load_checkpoint_and_dispatch


## Round 1 · Baseline prompt

In [2]:
import os
import pandas as pd

checkpoint_dir = "/workspace"

# Read while skipping malformed rows
cola_train_df = pd.read_csv(os.path.join(checkpoint_dir, "cola_train.csv"), on_bad_lines='skip')
cola_test_df = pd.read_csv(os.path.join(checkpoint_dir, "cola_test.csv"), on_bad_lines='skip')
sst2_train_df = pd.read_csv(os.path.join(checkpoint_dir, "sst2_train.csv"), on_bad_lines='skip')
sst2_test_df = pd.read_csv(os.path.join(checkpoint_dir, "sst2_test.csv"), on_bad_lines='skip')

# Convert
cola_train = cola_train_df.to_dict(orient='records')
cola_test = cola_test_df.to_dict(orient='records')
sst2_train = sst2_train_df.to_dict(orient='records')
sst2_test = sst2_test_df.to_dict(orient='records')

In [17]:
# Check results
print("SST-2 train set size:", len(sst2_train))  # should be 5000
print("SST-2 test set size:", len(sst2_test))    # should be 1000
print("CoLA train set size:", len(cola_train))  # should be 5000
print("CoLA test set size:", len(cola_test))    # should stay unchanged

SST-2 train set size: 5000
SST-2 test set size: 1000
CoLA train set size: 5000
CoLA test set size: 1000


In [18]:
import random

random.seed(42)
cola_test_200 = random.sample(cola_test, 200)
sst2_test_200 = random.sample(sst2_test, 200)

In [19]:
datasets = {
    "sst2": {"train": sst2_train, "test": sst2_test},
    "cola": {"train": cola_train, "test": cola_test}
}

In [22]:
models = [
    # ("TURKCELL/Turkcell-LLM-7b-v1", "turkcell")
    # ("ytu-ce-cosmos/Turkish-Llama-8b-DPO-v0.1", "cosmos")
    ("KOCDIGITAL/Kocdigital-LLM-8b-v0.1", "koc")
    # ("google/gemma-2-9b-it", "gemma")
]

In [21]:
def clear_model():
    gc.collect()
    torch.cuda.empty_cache()
    globals().pop('model', None)
    globals().pop('tokenizer', None)
    print("🚮 RAM and GPU memory cleared.")

def get_few_shot_prompt(dataset, sentence, shot, task):
    if shot == 0:
        if task == "cola":
            return f"Cümle: {sentence}\nBu cümle gramatik olarak doğru mu? Evet ya da Hayır ile cevap ver."
        else:
            return f"Cümle: {sentence}\nBu cümle olumlu mu, olumsuz mu? Cevabın sadece 'Olumlu' ya da 'Olumsuz' olsun."
    
    examples = random.sample(dataset, k=shot)
    prompt = ""
    for ex in examples:
        if task == "cola":
            label_str = "Evet" if ex["label"] == 1 else "Hayır"
        else:
            label_str = "Olumlu" if ex["label"] == 1 else "Olumsuz"
        prompt += f"Cümle: {ex['sentence']}\nCevap: {label_str}\n"
    
    prompt += f"Cümle: {sentence}\nCevap:"
    return prompt

def get_llm_prediction(prompt, model, tokenizer, max_tokens=10):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def parse_output(text, task="sst2"):
    text = text.lower()
    if task == "cola":
        if "evet" in text: return 1
        elif "hayır" in text: return 0
    else:
        if "olumlu" in text: return 1
        elif "olumsuz" in text: return 0
    return -1

In [8]:
gc.collect()
torch.cuda.empty_cache()

In [27]:
from tqdm import tqdm

shot_types = [0, 3, 5]
results = []

print(f"\n🎯 Starting! Selected models: {[name for _, name in models]}")
print(f"📚 Datasets: {list(datasets.keys())}")
print(f"🔁 Shot counts: {shot_types}")

for model_id, model_name in models:
    print(f"\n🚀 [{model_name.upper()}] downloading and loading model...")

    model_dir = f"/content/{model_name}_model"
    offload_dir = f"/content/offload_{model_name}"

    # 🔥 Download the model each time
    snapshot_download(
        repo_id=model_id,
        local_dir=model_dir,
        local_dir_use_symlinks=False
    )

    # 🧠 Download tokenizer and config
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    config = AutoConfig.from_pretrained(model_id)

    # 🚀 Load model
    with init_empty_weights():
        model = AutoModelForCausalLM.from_config(config)

    model = load_checkpoint_and_dispatch(
        model,
        model_dir,
        device_map="sequential",
        no_split_module_classes=["LlamaDecoderLayer"],
        offload_folder=offload_dir,
        dtype=torch.float16,
        offload_buffers=True,
        offload_state_dict=True
    )

    print(f"✅ [{model_name.upper()}] model downloaded and loaded.")

    for dataset_name, data in datasets.items():
        task_train = data["train"]
        all_examples = data["test"][:200]

        print(f"\n📂 Dataset: {dataset_name.upper()} ({len(all_examples)} examples)")

        for shot in shot_types:
            print(f"\n💡 Starting: {shot}-shot test (200 examples)")

            correct = 0
            total = 0
            predictions = []
            start_time = time.time()

            for example in tqdm(all_examples, desc=f"{model_name.upper()} | {dataset_name.upper()} | {shot}-shot"):
                prompt = get_few_shot_prompt(task_train, example["sentence"], shot=shot, task=dataset_name)
                try:
                    output = get_llm_prediction(prompt, model, tokenizer)
                    pred = parse_output(output, task=dataset_name)
                except Exception as e:
                    print(f"  ❌ Hata: {e}")
                    pred = -1

                gc.collect()
                torch.cuda.empty_cache()

                predictions.append({
                    "sentence": example["sentence"],
                    "label": example["label"],
                    "prediction": pred
                })

                if pred != -1:
                    total += 1
                    if pred == example["label"]:
                        correct += 1

            end_time = time.time()
            acc = correct / total if total > 0 else 0.0

            df_results = pd.DataFrame(predictions)
            df_results["correct"] = df_results["label"] == df_results["prediction"]
            df_results["model"] = model_name
            df_results["dataset"] = dataset_name
            df_results["shot"] = shot
            df_results["duration_sec"] = round(end_time - start_time, 2)

            out_path = f"/workspace/{model_name}_{dataset_name}_{shot}shot.csv"
            df_results.to_csv(out_path, index=False)

            print(f"✅ {shot}-shot done | Accuracy: {acc:.2%} | Time: {df_results['duration_sec'][0]} s")
            print(f"💾 Result saved: {out_path}")

    clear_model()
    print(f"\n🧹 [{model_name.upper()}] RAM and GPU cleared.")



🎯 Starting! Selected models: ['turkcell']
📚 Datasets: ['sst2', 'cola']
🔁 Shot counts: [0, 3, 5]

🚀 [TURKCELL] downloading and loading model...


✅ [TURKCELL] model downloaded and loaded.

📂 Dataset: SST2 (200 examples)

💡 Starting: 0-shot test (200 examples)


✅ 0-shot done | Accuracy: 74.11% | Time: 80.1 s
💾 Result saved: /workspace/turkcell_sst2_0shot.csv

💡 Starting: 3-shot test (200 examples)


✅ 3-shot done | Accuracy: 75.76% | Time: 82.21 s
💾 Result saved: /workspace/turkcell_sst2_3shot.csv

💡 Starting: 5-shot test (200 examples)


✅ 5-shot done | Accuracy: 72.73% | Time: 84.42 s
💾 Result saved: /workspace/turkcell_sst2_5shot.csv

📂 Dataset: COLA (200 examples)

💡 Starting: 0-shot test (200 examples)


✅ 0-shot done | Accuracy: 39.00% | Time: 78.12 s
💾 Result saved: /workspace/turkcell_cola_0shot.csv

💡 Starting: 3-shot test (200 examples)


✅ 3-shot done | Accuracy: 41.00% | Time: 79.07 s
💾 Result saved: /workspace/turkcell_cola_3shot.csv

💡 Starting: 5-shot test (200 examples)


✅ 5-shot done | Accuracy: 41.50% | Time: 79.84 s
💾 Result saved: /workspace/turkcell_cola_5shot.csv
🚮 RAM and GPU memory cleared.

🧹 [TURKCELL] RAM and GPU cleared.


In [36]:
from tqdm import tqdm
import os
import time
import gc
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from accelerate import init_empty_weights, load_checkpoint_and_dispatch
from huggingface_hub import snapshot_download

# Setup
shot_types = [0, 3, 5]
results = []

print(f"\n🎯 Starting! Selected models: {[name for _, name in models]}")
print(f"📚 Datasets: {list(datasets.keys())}")
print(f"🔁 Shot counts: {shot_types}")

# Model loop
for model_id, model_name in models:
    print(f"\n🚀 [{model_name.upper()}] downloading and loading model...")

    model_dir = f"/content/{model_name}_model"
    offload_dir = f"/content/offload_{model_name}"

    # Download model
    snapshot_download(
        repo_id=model_id,
        local_dir=model_dir,
        local_dir_use_symlinks=False
    )

    # Tokenizer and config
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    config = AutoConfig.from_pretrained(model_id)

    # Load model
    with init_empty_weights():
        model = AutoModelForCausalLM.from_config(config)

    model = load_checkpoint_and_dispatch(
        model,
        model_dir,
        device_map="sequential",
        no_split_module_classes=["LlamaDecoderLayer"],
        offload_folder=offload_dir,
        dtype=torch.float16,
        offload_buffers=True,
        offload_state_dict=True
    )

    print(f"✅ [{model_name.upper()}] model downloaded and loaded.")

    # Dataset loop
    for dataset_name, data in datasets.items():
        task_train = data["train"]
        all_examples = data["test"][:200]

        print(f"\n📂 Dataset: {dataset_name.upper()} ({len(all_examples)} examples)")

        for shot in shot_types:
            print(f"\n💡 Starting: {shot}-shot test")

            correct = 0
            total = 0
            predictions = []
            start_time = time.time()

            for example in tqdm(all_examples, desc=f"{model_name.upper()} | {dataset_name.upper()} | {shot}-shot"):
                prompt = get_few_shot_prompt(task_train, example["sentence"], shot=shot, task=dataset_name)
                try:
                    output = get_llm_prediction(prompt, model, tokenizer)
                    pred = parse_output(output, task=dataset_name)
                except Exception as e:
                    print(f"  ❌ Hata: {e}")
                    pred = -1

                gc.collect()
                torch.cuda.empty_cache()

                predictions.append({
                    "sentence": example["sentence"],
                    "label": example["label"],
                    "prediction": pred
                })

                if pred != -1:
                    total += 1
                    if pred == example["label"]:
                        correct += 1

            end_time = time.time()
            acc = correct / total if total > 0 else 0.0

            df_results = pd.DataFrame(predictions)
            df_results["correct"] = df_results["label"] == df_results["prediction"]
            df_results["model"] = model_name
            df_results["dataset"] = dataset_name
            df_results["shot"] = shot
            df_results["duration_sec"] = round(end_time - start_time, 2)

            # Save results
            out_path = f"/workspace/{model_name}_{dataset_name}_{shot}shot.csv"
            df_results.to_csv(out_path, index=False)

            print(f"✅ {shot}-shot done | Accuracy: {acc:.2%} | Time: {df_results['duration_sec'][0]} s")
            print(f"💾 Result saved: {out_path}")

            # Add summary to results
            results.append({
                "model": model_name,
                "dataset": dataset_name,
                "shot": shot,
                "accuracy": acc,
                "correct": correct,
                "total": total,
                "duration_sec": round(end_time - start_time, 2)
            })

    # Clean up after model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\n🧹 [{model_name.upper()}] RAM and GPU cleared.")

# Save summary output of all models
results_df = pd.DataFrame(results)
results_df.to_csv("/workspace/llm_cosmos_results_summary.csv", index=False)
print("\n📄 Summary results saved → /workspace/llm_cosmos_results_summary.csv")



🎯 Starting! Selected models: ['cosmos']
📚 Datasets: ['sst2', 'cola']
🔁 Shot counts: [0, 3, 5]

🚀 [COSMOS] downloading and loading model...


✅ [COSMOS] model downloaded and loaded.

📂 Dataset: SST2 (200 examples)

💡 Starting: 0-shot test


✅ 0-shot done | Accuracy: 74.49% | Time: 63.27 s
💾 Result saved: /workspace/cosmos_sst2_0shot.csv

💡 Starting: 3-shot test


✅ 3-shot done | Accuracy: 73.33% | Time: 96.17 s
💾 Result saved: /workspace/cosmos_sst2_3shot.csv

💡 Starting: 5-shot test


✅ 5-shot done | Accuracy: 73.37% | Time: 90.73 s
💾 Result saved: /workspace/cosmos_sst2_5shot.csv

📂 Dataset: COLA (200 examples)

💡 Starting: 0-shot test


✅ 0-shot done | Accuracy: 39.00% | Time: 82.77 s
💾 Result saved: /workspace/cosmos_cola_0shot.csv

💡 Starting: 3-shot test


✅ 3-shot done | Accuracy: 38.50% | Time: 82.95 s
💾 Result saved: /workspace/cosmos_cola_3shot.csv

💡 Starting: 5-shot test


✅ 5-shot done | Accuracy: 39.00% | Time: 88.22 s
💾 Result saved: /workspace/cosmos_cola_5shot.csv

🧹 [COSMOS] RAM and GPU cleared.

📄 Summary results saved → /workspace/llm_cosmos_results_summary.csv


In [42]:
from tqdm import tqdm
import os
import time
import gc
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from accelerate import init_empty_weights, load_checkpoint_and_dispatch
from huggingface_hub import snapshot_download

# Setup
shot_types = [0, 3, 5]
results = []

print(f"\n🎯 Starting! Selected models: {[name for _, name in models]}")
print(f"📚 Datasets: {list(datasets.keys())}")
print(f"🔁 Shot counts: {shot_types}")

# Model loop
for model_id, model_name in models:
    print(f"\n🚀 [{model_name.upper()}] downloading and loading model...")

    model_dir = f"/content/{model_name}_model"
    offload_dir = f"/content/offload_{model_name}"

    # Download model
    snapshot_download(
        repo_id=model_id,
        local_dir=model_dir,
        local_dir_use_symlinks=False
    )

    # Tokenizer and config
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    config = AutoConfig.from_pretrained(model_id)

    # Load model
    with init_empty_weights():
        model = AutoModelForCausalLM.from_config(config)

    model = load_checkpoint_and_dispatch(
        model,
        model_dir,
        device_map="sequential",
        no_split_module_classes=["LlamaDecoderLayer"],
        offload_folder=offload_dir,
        dtype=torch.float16,
        offload_buffers=True,
        offload_state_dict=True
    )

    print(f"✅ [{model_name.upper()}] model downloaded and loaded.")

    # Dataset loop
    for dataset_name, data in datasets.items():
        task_train = data["train"]
        all_examples = data["test"][:200]

        print(f"\n📂 Dataset: {dataset_name.upper()} ({len(all_examples)} examples)")

        for shot in shot_types:
            print(f"\n💡 Starting: {shot}-shot test")

            correct = 0
            total = 0
            predictions = []
            start_time = time.time()

            for example in tqdm(all_examples, desc=f"{model_name.upper()} | {dataset_name.upper()} | {shot}-shot"):
                prompt = get_few_shot_prompt(task_train, example["sentence"], shot=shot, task=dataset_name)
                try:
                    output = get_llm_prediction(prompt, model, tokenizer)
                    pred = parse_output(output, task=dataset_name)
                except Exception as e:
                    print(f"  ❌ Hata: {e}")
                    pred = -1

                gc.collect()
                torch.cuda.empty_cache()

                predictions.append({
                    "sentence": example["sentence"],
                    "label": example["label"],
                    "prediction": pred
                })

                if pred != -1:
                    total += 1
                    if pred == example["label"]:
                        correct += 1

            end_time = time.time()
            acc = correct / total if total > 0 else 0.0

            df_results = pd.DataFrame(predictions)
            df_results["correct"] = df_results["label"] == df_results["prediction"]
            df_results["model"] = model_name
            df_results["dataset"] = dataset_name
            df_results["shot"] = shot
            df_results["duration_sec"] = round(end_time - start_time, 2)

            # Save results
            out_path = f"/workspace/{model_name}_{dataset_name}_{shot}shot.csv"
            df_results.to_csv(out_path, index=False)

            print(f"✅ {shot}-shot done | Accuracy: {acc:.2%} | Time: {df_results['duration_sec'][0]} s")
            print(f"💾 Result saved: {out_path}")

            # Add summary to results
            results.append({
                "model": model_name,
                "dataset": dataset_name,
                "shot": shot,
                "accuracy": acc,
                "correct": correct,
                "total": total,
                "duration_sec": round(end_time - start_time, 2)
            })

    # Clean up after model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\n🧹 [{model_name.upper()}] RAM and GPU cleared.")

# Save summary output of all models
results_df = pd.DataFrame(results)
results_df.to_csv("/workspace/llm_koc_results_summary.csv", index=False)
print("\n📄 Summary results saved → /workspace/llm_koc_results_summary.csv")



🎯 Starting! Selected models: ['koc']
📚 Datasets: ['sst2', 'cola']
🔁 Shot counts: [0, 3, 5]

🚀 [KOC] downloading and loading model...


✅ [KOC] model downloaded and loaded.

📂 Dataset: SST2 (200 examples)

💡 Starting: 0-shot test


✅ 0-shot done | Accuracy: 74.49% | Time: 80.23 s
💾 Result saved: /workspace/koc_sst2_0shot.csv

💡 Starting: 3-shot test


✅ 3-shot done | Accuracy: 71.94% | Time: 85.94 s
💾 Result saved: /workspace/koc_sst2_3shot.csv

💡 Starting: 5-shot test


✅ 5-shot done | Accuracy: 75.00% | Time: 90.01 s
💾 Result saved: /workspace/koc_sst2_5shot.csv

📂 Dataset: COLA (200 examples)

💡 Starting: 0-shot test


✅ 0-shot done | Accuracy: 39.00% | Time: 85.17 s
💾 Result saved: /workspace/koc_cola_0shot.csv

💡 Starting: 3-shot test


✅ 3-shot done | Accuracy: 38.50% | Time: 86.28 s
💾 Result saved: /workspace/koc_cola_3shot.csv

💡 Starting: 5-shot test


✅ 5-shot done | Accuracy: 42.00% | Time: 86.82 s
💾 Result saved: /workspace/koc_cola_5shot.csv

🧹 [KOC] RAM and GPU cleared.

📄 Summary results saved → /workspace/llm_koc_results_summary.csv


In [49]:
from huggingface_hub import login
login()  # asks for your Hugging Face token (or set the HF_TOKEN environment variable)

In [53]:
# Model loading updated: using `from_pretrained` fully fixes the `meta tensor` error

from tqdm import tqdm
import os
import time
import gc
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download

# Setup
shot_types = [0, 3, 5]
results = []

print(f"\n🎯 Starting! Selected models: {[name for _, name in models]}")
print(f"📚 Datasets: {list(datasets.keys())}")
print(f"🔁 Shot counts: {shot_types}")

# Model loop
for model_id, model_name in models:
    print(f"\n🚀 [{model_name.upper()}] downloading and loading model...")

    model_dir = f"/content/{model_name}_model"
    offload_dir = f"/content/offload_{model_name}"

    # Download model
    snapshot_download(
        repo_id=model_id,
        local_dir=model_dir,
        local_dir_use_symlinks=False
    )

    # Tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        device_map="auto",
        torch_dtype=torch.float16
    )

    print(f"✅ [{model_name.upper()}] model downloaded and loaded.")

    # Dataset loop
    for dataset_name, data in datasets.items():
        task_train = data["train"]
        all_examples = data["test"][:200]

        print(f"\n📂 Dataset: {dataset_name.upper()} ({len(all_examples)} examples)")

        for shot in shot_types:
            print(f"\n💡 Starting: {shot}-shot test")

            correct = 0
            total = 0
            predictions = []
            start_time = time.time()

            for example in tqdm(all_examples, desc=f"{model_name.upper()} | {dataset_name.upper()} | {shot}-shot"):
                prompt = get_few_shot_prompt(task_train, example["sentence"], shot=shot, task=dataset_name)
                try:
                    output = get_llm_prediction(prompt, model, tokenizer)
                    pred = parse_output(output, task=dataset_name)
                except Exception as e:
                    print(f"  ❌ Hata: {e}")
                    pred = -1

                gc.collect()
                torch.cuda.empty_cache()

                predictions.append({
                    "sentence": example["sentence"],
                    "label": example["label"],
                    "prediction": pred
                })

                if pred != -1:
                    total += 1
                    if pred == example["label"]:
                        correct += 1

            end_time = time.time()
            acc = correct / total if total > 0 else 0.0

            df_results = pd.DataFrame(predictions)
            df_results["correct"] = df_results["label"] == df_results["prediction"]
            df_results["model"] = model_name
            df_results["dataset"] = dataset_name
            df_results["shot"] = shot
            df_results["duration_sec"] = round(end_time - start_time, 2)

            # Save results
            out_path = f"/workspace/{model_name}_{dataset_name}_{shot}shot.csv"
            df_results.to_csv(out_path, index=False)

            print(f"✅ {shot}-shot done | Accuracy: {acc:.2%} | Time: {df_results['duration_sec'][0]} s")
            print(f"💾 Result saved: {out_path}")

            # Add summary to results
            results.append({
                "model": model_name,
                "dataset": dataset_name,
                "shot": shot,
                "accuracy": acc,
                "correct": correct,
                "total": total,
                "duration_sec": round(end_time - start_time, 2)
            })

    # Clean up after model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\n🧹 [{model_name.upper()}] RAM and GPU cleared.")

# Save summary output of all models
results_df = pd.DataFrame(results)
results_df.to_csv("/workspace/llm_gemma_results_summary.csv", index=False)
print("\n📄 Summary results saved → /workspace/llm_gemma_results_summary.csv")



🎯 Starting! Selected models: ['gemma']
📚 Datasets: ['sst2', 'cola']
🔁 Shot counts: [0, 3, 5]

🚀 [GEMMA] downloading and loading model...


✅ [GEMMA] model downloaded and loaded.

📂 Dataset: SST2 (200 examples)

💡 Starting: 0-shot test


✅ 0-shot done | Accuracy: 74.49% | Time: 109.42 s
💾 Result saved: /workspace/gemma_sst2_0shot.csv

💡 Starting: 3-shot test


✅ 3-shot done | Accuracy: 72.36% | Time: 141.28 s
💾 Result saved: /workspace/gemma_sst2_3shot.csv

💡 Starting: 5-shot test


✅ 5-shot done | Accuracy: 74.37% | Time: 139.6 s
💾 Result saved: /workspace/gemma_sst2_5shot.csv

📂 Dataset: COLA (200 examples)

💡 Starting: 0-shot test


✅ 0-shot done | Accuracy: 39.00% | Time: 138.08 s
💾 Result saved: /workspace/gemma_cola_0shot.csv

💡 Starting: 3-shot test


✅ 3-shot done | Accuracy: 41.00% | Time: 139.0 s
💾 Result saved: /workspace/gemma_cola_3shot.csv

💡 Starting: 5-shot test


✅ 5-shot done | Accuracy: 40.50% | Time: 136.0 s
💾 Result saved: /workspace/gemma_cola_5shot.csv

🧹 [GEMMA] RAM and GPU cleared.

📄 Summary results saved → /workspace/llm_gemma_results_summary.csv


### Improved prompt and output parsing, plus balanced datasets

In [54]:
from collections import Counter

def print_label_balance(name, dataset):
    labels = [ex["label"] for ex in dataset]
    counter = Counter(labels)
    total = sum(counter.values())
    print(f"\n📊 {name} Label distribution:")
    for label, count in counter.items():
        print(f"  Label {label}: {count} examples ({count / total:.2%})")

# Uygula
print_label_balance("CoLA Train", cola_train)
print_label_balance("CoLA Test", cola_test)
print_label_balance("SST-2 Train", sst2_train)
print_label_balance("SST-2 Test", sst2_test)


📊 CoLA Train Label distribution:
  Label 0: 3149 examples (62.98%)
  Label 1: 1851 examples (37.02%)

📊 CoLA Test Label distribution:
  Label 1: 379 examples (37.90%)
  Label 0: 621 examples (62.10%)

📊 SST-2 Train Label distribution:
  Label 1: 3642 examples (72.84%)
  Label 0: 1358 examples (27.16%)

📊 SST-2 Test Label distribution:
  Label 1: 708 examples (70.80%)
  Label 0: 292 examples (29.20%)


**Why balancing matters:**

- On SST-2, always answering 1 (positive) looks reasonable because the data is skewed → the model becomes biased.
- On CoLA the same happens in reverse → it tends to always answer "grammatically incorrect".

**Result:** high accuracy but low F1 (the model looks good just by predicting the majority class). The few-shot examples are affected by this imbalance as well.

In [55]:
from datasets import load_dataset
import random

def get_balanced_subset(dataset, total_count):
    label_0 = [ex for ex in dataset if ex["label"] == 0]
    label_1 = [ex for ex in dataset if ex["label"] == 1]
    n = total_count // 2
    balanced = []
    for i in range(n):
        balanced.append(random.choice(label_0))
        balanced.append(random.choice(label_1))
    random.shuffle(balanced)
    return balanced

# 1. Load the data
ds_cola = load_dataset("turkish-nlp-suite/TrGLUE", "cola")
ds_sst2 = load_dataset("turkish-nlp-suite/TrGLUE", "sst2")

# 2. Create balanced train and test sets
cola_train = get_balanced_subset(ds_cola["train"], 5000)
cola_test  = get_balanced_subset(ds_cola["test"], 1000)

sst2_train = get_balanced_subset(ds_sst2["train"], 5000)
sst2_test  = get_balanced_subset(ds_sst2["test"], 1000)


In [59]:
# Uygula
print_label_balance("CoLA Train", cola_train)
print_label_balance("CoLA Test", cola_test)
print_label_balance("SST-2 Train", sst2_train)
print_label_balance("SST-2 Test", sst2_test)


📊 CoLA Train Label distribution:
  Label 1: 2500 examples (50.00%)
  Label 0: 2500 examples (50.00%)

📊 CoLA Test Label distribution:
  Label 0: 500 examples (50.00%)
  Label 1: 500 examples (50.00%)

📊 SST-2 Train Label distribution:
  Label 0: 2500 examples (50.00%)
  Label 1: 2500 examples (50.00%)

📊 SST-2 Test Label distribution:
  Label 0: 500 examples (50.00%)
  Label 1: 500 examples (50.00%)


In [60]:
# Convert CoLA training data to a pandas DataFrame
cola_train_df = pd.DataFrame(cola_train)
cola_test_df = pd.DataFrame(cola_test)

# Convert SST-2 training data to a pandas DataFrame
sst2_train_df = pd.DataFrame(sst2_train)
sst2_test_df = pd.DataFrame(sst2_test)

In [61]:
# Save data as CSV to the given directory
cola_train_df.to_csv(os.path.join(checkpoint_dir, "cola_train2.csv"), index=False)  # CoLA train set
cola_test_df.to_csv(os.path.join(checkpoint_dir, "cola_test2.csv"), index=False)    # CoLA test set

sst2_train_df.to_csv(os.path.join(checkpoint_dir, "sst2_train2.csv"), index=False)  # SST-2 train set
sst2_test_df.to_csv(os.path.join(checkpoint_dir, "sst2_test2.csv"), index=False)    # SST-2 test set

# Done
print(f"Data saved to '{checkpoint_dir}' as CSV.")

Data saved to '/workspace' as CSV.


In [14]:
import os
import pandas as pd

checkpoint_dir = "/workspace"

# Read while skipping malformed rows
cola_train_df = pd.read_csv(os.path.join(checkpoint_dir, "cola_train2.csv"), on_bad_lines='skip')
cola_test_df = pd.read_csv(os.path.join(checkpoint_dir, "cola_test2.csv"), on_bad_lines='skip')
sst2_train_df = pd.read_csv(os.path.join(checkpoint_dir, "sst2_train2.csv"), on_bad_lines='skip')
sst2_test_df = pd.read_csv(os.path.join(checkpoint_dir, "sst2_test2.csv"), on_bad_lines='skip')

# Convert
cola_train = cola_train_df.to_dict(orient='records')
cola_test = cola_test_df.to_dict(orient='records')
sst2_train = sst2_train_df.to_dict(orient='records')
sst2_test = sst2_test_df.to_dict(orient='records')

In [15]:
# Check results
print("SST-2 train set size:", len(sst2_train))  # should be 5000
print("SST-2 test set size:", len(sst2_test))    # should be 1000
print("CoLA train set size:", len(cola_train))  # should be 5000
print("CoLA test set size:", len(cola_test))    # should stay unchanged

SST-2 train set size: 5000
SST-2 test set size: 1000
CoLA train set size: 5000
CoLA test set size: 1000


In [16]:
datasets = {
    "sst2": {"train": sst2_train, "test": sst2_test},
    "cola": {"train": cola_train, "test": cola_test}
}

In [4]:
def get_few_shot_prompt(dataset, sentence, shot, task):
    prompt = ""

    if task == "cola":
        prompt += "Aşağıdaki cümlelerin gramer açısından doğru olup olmadığını değerlendir.\n\n"
    elif task == "sst2":
        prompt += "Aşağıdaki cümlelerin duygu durumunu değerlendir. Cevabın sadece 'Olumlu' ya da 'Olumsuz' olsun.\n\n"

    if shot > 0:
        examples = random.sample(dataset, k=shot)
        for ex in examples:
            if task == "cola":
                label_str = "Evet" if ex["label"] == 1 else "Hayır"
            else:
                label_str = "Olumlu" if ex["label"] == 1 else "Olumsuz"
            prompt += f"Cümle: {ex['sentence']}\nCevap: {label_str}\n\n"

    # Test example
    prompt += f"Cümle: {sentence}\nCevap:"
    return prompt

def parse_output(text, task="sst2"):
    text = text.lower().strip()

    # Take the last answer
    if "cevap:" in text:
        text = text.split("cevap:")[-1].strip()

    if task == "cola":
        if "evet" in text[:20]: return 1
        elif "hayır" in text[:20]: return 0
    else:
        if "olumlu" in text[:20]: return 1
        elif "olumsuz" in text[:20]: return 0
        elif "1" == text: return 1
        elif "0" == text: return 0

    return -1

In [70]:
# Model loading updated: using `from_pretrained` fully fixes the `meta tensor` error

from tqdm import tqdm
import os
import time
import gc
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download

# Setup
shot_types = [0, 3, 5]
results = []

print(f"\n🎯 Starting! Selected models: {[name for _, name in models]}")
print(f"📚 Datasets: {list(datasets.keys())}")
print(f"🔁 Shot counts: {shot_types}")

# Model loop
for model_id, model_name in models:
    print(f"\n🚀 [{model_name.upper()}] downloading and loading model...")

    model_dir = f"/content/{model_name}_model"
    offload_dir = f"/content/offload_{model_name}"

    # Download model
    snapshot_download(
        repo_id=model_id,
        local_dir=model_dir,
        local_dir_use_symlinks=False
    )

    # Tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        device_map="auto",
        torch_dtype=torch.float16
    )

    print(f"✅ [{model_name.upper()}] model downloaded and loaded.")

    # Dataset loop
    for dataset_name, data in datasets.items():
        task_train = data["train"]
        all_examples = data["test"][:200]

        print(f"\n📂 Dataset: {dataset_name.upper()} ({len(all_examples)} examples)")

        for shot in shot_types:
            print(f"\n💡 Starting: {shot}-shot test")

            correct = 0
            total = 0
            predictions = []
            start_time = time.time()

            for example in tqdm(all_examples, desc=f"{model_name.upper()} | {dataset_name.upper()} | {shot}-shot"):
                prompt = get_few_shot_prompt(task_train, example["sentence"], shot=shot, task=dataset_name)
                try:
                    output = get_llm_prediction(prompt, model, tokenizer)
                    pred = parse_output(output, task=dataset_name)
                except Exception as e:
                    print(f"  ❌ Hata: {e}")
                    pred = -1

                gc.collect()
                torch.cuda.empty_cache()

                predictions.append({
                    "sentence": example["sentence"],
                    "label": example["label"],
                    "prediction": pred
                })

                if pred != -1:
                    total += 1
                    if pred == example["label"]:
                        correct += 1

            end_time = time.time()
            acc = correct / total if total > 0 else 0.0

            df_results = pd.DataFrame(predictions)
            df_results["correct"] = df_results["label"] == df_results["prediction"]
            df_results["model"] = model_name
            df_results["dataset"] = dataset_name
            df_results["shot"] = shot
            df_results["duration_sec"] = round(end_time - start_time, 2)

            # Save results
            out_path = f"/workspace/{model_name}_{dataset_name}_{shot}shot_2.csv"
            df_results.to_csv(out_path, index=False)

            print(f"✅ {shot}-shot done | Accuracy: {acc:.2%} | Time: {df_results['duration_sec'][0]} s")
            print(f"💾 Result saved: {out_path}")

            # Add summary to results
            results.append({
                "model": model_name,
                "dataset": dataset_name,
                "shot": shot,
                "accuracy": acc,
                "correct": correct,
                "total": total,
                "duration_sec": round(end_time - start_time, 2)
            })

    # Clean up after model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\n🧹 [{model_name.upper()}] RAM and GPU cleared.")

# Save summary output of all models
results_df = pd.DataFrame(results)
results_df.to_csv("/workspace/llm_gemma_results2_summary.csv", index=False)
print("\n📄 Summary results saved → /workspace/llm_gemma_results2_summary.csv")



🎯 Starting! Selected models: ['gemma']
📚 Datasets: ['sst2', 'cola']
🔁 Shot counts: [0, 3, 5]

🚀 [GEMMA] downloading and loading model...


✅ [GEMMA] model downloaded and loaded.

📂 Dataset: SST2 (200 examples)

💡 Starting: 0-shot test


✅ 0-shot done | Accuracy: 79.29% | Time: 149.0 s
💾 Result saved: /workspace/gemma_sst2_0shot_2.csv

💡 Starting: 3-shot test


✅ 3-shot done | Accuracy: 79.90% | Time: 175.26 s
💾 Result saved: /workspace/gemma_sst2_3shot_2.csv

💡 Starting: 5-shot test


✅ 5-shot done | Accuracy: 65.14% | Time: 177.49 s
💾 Result saved: /workspace/gemma_sst2_5shot_2.csv

📂 Dataset: COLA (200 examples)

💡 Starting: 0-shot test


✅ 0-shot done | Accuracy: 85.71% | Time: 181.54 s
💾 Result saved: /workspace/gemma_cola_0shot_2.csv

💡 Starting: 3-shot test


✅ 3-shot done | Accuracy: 63.00% | Time: 183.01 s
💾 Result saved: /workspace/gemma_cola_3shot_2.csv

💡 Starting: 5-shot test


✅ 5-shot done | Accuracy: 65.00% | Time: 182.75 s
💾 Result saved: /workspace/gemma_cola_5shot_2.csv

🧹 [GEMMA] RAM and GPU cleared.

📄 Summary results saved → /workspace/llm_gemma_results2_summary.csv


In [5]:
# def parse_output(text, task="cola"):
#     original = text
#     text = text.lower().strip()

#     if "cevap:" in text:
#         text = text.split("cevap:")[-1].strip()

#     if task == "cola":
#         if "doğru" in text[:200] or "gramer açısından uygun" in text: return 1
#         elif "yanlış" in text[:200] or "hatalı" in text: return 0
#         elif "evet" in text[:200]: return 1
#         elif "hayır" in text[:200]: return 0
#     else:
#         if "olumlu" in text[:200]: return 1
#         elif "olumsuz" in text[:200]: return 0
#         elif text.strip() == "1": return 1
#         elif text.strip() == "0": return 0

#     print("❌ parse_output() could not parse:", original)
#     return -1
def get_few_shot_prompt(dataset, sentence, shot, task):
    prompt = ""

    if task == "cola":
        prompt += "Aşağıdaki cümlelerin gramer açısından doğru olup olmadığını değerlendir. Cevabın sadece 'Doğru' ya da 'Yanlış' olsun.\n\n"
    elif task == "sst2":
        prompt += "Aşağıdaki cümlelerin duygu durumunu değerlendir. Cevabın sadece 'Olumlu' ya da 'Olumsuz' olsun.\n\n"

    if shot > 0:
        examples = random.sample(dataset, k=shot)
        for ex in examples:
            if task == "cola":
                label_str = "Doğru" if ex["label"] == 1 else "Yanlış"
            else:
                label_str = "Olumlu" if ex["label"] == 1 else "Olumsuz"
            prompt += f"Cümle: {ex['sentence']}\nCevap: {label_str}\n\n"

    # Test sentence
    prompt += f"Cümle: {sentence}\nCevap:"
    return prompt
    
# def parse_output(text, task="cola"):
#     original = text
#     text = text.lower().strip()

# # If it contains "cevap:", keep only the last part
#     if "cevap:" in text:
#         text = text.split("cevap:")[-1].strip()
#     else:
# # fallback: the answer may already be a single line
#         text = text.strip()

#     if task == "cola":
#         if (
#             text.startswith("evet")
#             or "doğru" in text[:40]
#             or "gramer açısından uygun" in text
#         ):
#             return 1
#         elif (
#             text.startswith("hayır")
#             or "yanlış" in text[:40]
#             or "hata" in text[:40]
#             or "hatalı" in text[:40]
#             or "gramer hatası" in text
#             or "hata var" in text
#         ):
#             return 0

#     elif task == "sst2":
#         if text.startswith("olumlu"):
#             return 1
#         elif text.startswith("olumsuz") or text.startswith("kötü") or "olumsuz" in text[:40]:
#             return 0
#         elif text.strip() == "1":
#             return 1
#         elif text.strip() == "0":
#             return 0

#     print("❌ parse_output() could not parse:", original)
#     return -1

def parse_output(text, task="cola"):
    original = text
    text = text.lower().strip()

    # If it contains "cevap:", keep only the last part
    if "cevap:" in text:
        text = text.split("cevap:")[-1].strip()
    else:
        text = text.strip()

    if task == "cola":
        if text.startswith("doğru"):
            return 1
        elif text.startswith("yanlış"):
            return 0

    elif task == "sst2":
        if text.startswith("olumlu"):
            return 1
        elif text.startswith("olumsuz") or "olumsuz" in text[:40]:
            return 0
        elif text.strip() == "1":
            return 1
        elif text.strip() == "0":
            return 0

    print("❌ parse_output() could not parse:", original)
    return -1

def get_llm_prediction(prompt, model, tokenizer, max_tokens=20):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            temperature=0.0,
            repetition_penalty=1.1,
            early_stopping=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [26]:
def get_few_shot_prompt(dataset, sentence, shot, task):
    prompt = ""

    if task == "cola":
        prompt += "Aşağıdaki cümlelerin gramer açısından doğru olup olmadığını değerlendir.\n\n"
    elif task == "sst2":
        prompt += "Aşağıdaki cümlelerin duygu durumunu değerlendir. Cevabın sadece 'Olumlu' ya da 'Olumsuz' olsun.\n\n"

    if shot > 0:
        examples = random.sample(dataset, k=shot)
        for ex in examples:
            if task == "cola":
                label_str = "Evet" if ex["label"] == 1 else "Hayır"
            else:
                label_str = "Olumlu" if ex["label"] == 1 else "Olumsuz"
            prompt += f"Cümle: {ex['sentence']}\nCevap: {label_str}\n\n"

    # Test example
    prompt += f"Cümle: {sentence}\nCevap:"
    return prompt

def parse_output(text, task="sst2"):
    text = text.lower().strip()

    # Take the last answer
    if "cevap:" in text:
        text = text.split("cevap:")[-1].strip()

    if task == "cola":
        if "evet" in text[:20]: return 1
        elif "hayır" in text[:20]: return 0
    else:
        if "olumlu" in text[:20]: return 1
        elif "olumsuz" in text[:20]: return 0
        elif "1" == text: return 1
        elif "0" == text: return 0

    return -1

# def parse_output(text, task="sst2"):
#     original = text
#     text = text.lower().strip()

# # answer part only
#     if "cevap:" in text:
#         text = text.split("cevap:")[-1].strip()
#     else:
#         text = text.strip()

#     if task == "cola":
#         if text.startswith("doğru"): return 1
#         elif text.startswith("yanlış"): return 0
#     elif task == "sst2":
#         if text.startswith("olumlu"): return 1
#         elif text.startswith("olumsuz"): return 0
#         elif text.strip() == "1": return 1
#         elif text.strip() == "0": return 0

#     print("❌ parse_output could not parse:\n", original)
#     return -1


In [19]:
# Model loading updated: using `from_pretrained` fully fixes the `meta tensor` error

from tqdm import tqdm
import os
import time
import gc
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download

# Setup
shot_types = [0, 3, 5]
results = []

print(f"\n🎯 Starting! Selected models: {[name for _, name in models]}")
print(f"📚 Datasets: {list(datasets.keys())}")
print(f"🔁 Shot counts: {shot_types}")

# Model loop
for model_id, model_name in models:
    print(f"\n🚀 [{model_name.upper()}] downloading and loading model...")

    model_dir = f"/content/{model_name}_model"
    offload_dir = f"/content/offload_{model_name}"

    # Download model
    snapshot_download(
        repo_id=model_id,
        local_dir=model_dir,
        local_dir_use_symlinks=False
    )

    # Tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        device_map="auto",
        torch_dtype=torch.float16
    )

    print(f"✅ [{model_name.upper()}] model downloaded and loaded.")

    # Dataset loop
    for dataset_name, data in datasets.items():
        task_train = data["train"]
        all_examples = data["test"][:200]

        print(f"\n📂 Dataset: {dataset_name.upper()} ({len(all_examples)} examples)")

        for shot in shot_types:
            print(f"\n💡 Starting: {shot}-shot test")

            correct = 0
            total = 0
            predictions = []
            start_time = time.time()

            for example in tqdm(all_examples, desc=f"{model_name.upper()} | {dataset_name.upper()} | {shot}-shot"):
                prompt = get_few_shot_prompt(task_train, example["sentence"], shot=shot, task=dataset_name)
                try:
                    output = get_llm_prediction(prompt, model, tokenizer)
                    pred = parse_output(output, task=dataset_name)
                except Exception as e:
                    print(f"  ❌ Hata: {e}")
                    pred = -1

                gc.collect()
                torch.cuda.empty_cache()

                predictions.append({
                    "sentence": example["sentence"],
                    "label": example["label"],
                    "prediction": pred
                })

                if pred != -1:
                    total += 1
                    if pred == example["label"]:
                        correct += 1

            end_time = time.time()
            acc = correct / total if total > 0 else 0.0

            df_results = pd.DataFrame(predictions)
            df_results["correct"] = df_results["label"] == df_results["prediction"]
            df_results["model"] = model_name
            df_results["dataset"] = dataset_name
            df_results["shot"] = shot
            df_results["duration_sec"] = round(end_time - start_time, 2)

            # Save results
            out_path = f"/workspace/{model_name}_{dataset_name}_{shot}shot_2.csv"
            df_results.to_csv(out_path, index=False)

            print(f"✅ {shot}-shot done | Accuracy: {acc:.2%} | Time: {df_results['duration_sec'][0]} s")
            print(f"💾 Result saved: {out_path}")

            # Add summary to results
            results.append({
                "model": model_name,
                "dataset": dataset_name,
                "shot": shot,
                "accuracy": acc,
                "correct": correct,
                "total": total,
                "duration_sec": round(end_time - start_time, 2)
            })

    # Clean up after model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\n🧹 [{model_name.upper()}] RAM and GPU cleared.")

# Save summary output of all models
results_df = pd.DataFrame(results)
results_df.to_csv("/workspace/llm_cosmos_results2_summary.csv", index=False)
print("\n📄 Summary results saved → /workspace/llm_cosmos_results2_summary.csv")


🎯 Starting! Selected models: ['cosmos']
📚 Datasets: ['sst2', 'cola']
🔁 Shot counts: [0, 3, 5]

🚀 [COSMOS] downloading and loading model...


✅ [COSMOS] model downloaded and loaded.

📂 Dataset: SST2 (200 examples)

💡 Starting: 0-shot test


✅ 0-shot done | Accuracy: 80.51% | Time: 72.15 s
💾 Result saved: /workspace/cosmos_sst2_0shot_2.csv

💡 Starting: 3-shot test


✅ 3-shot done | Accuracy: 76.14% | Time: 127.64 s
💾 Result saved: /workspace/cosmos_sst2_3shot_2.csv

💡 Starting: 5-shot test


✅ 5-shot done | Accuracy: 67.02% | Time: 128.68 s
💾 Result saved: /workspace/cosmos_sst2_5shot_2.csv

📂 Dataset: COLA (200 examples)

💡 Starting: 0-shot test


✅ 0-shot done | Accuracy: 0.00% | Time: 123.2 s
💾 Result saved: /workspace/cosmos_cola_0shot_2.csv

💡 Starting: 3-shot test


✅ 3-shot done | Accuracy: 55.96% | Time: 121.55 s
💾 Result saved: /workspace/cosmos_cola_3shot_2.csv

💡 Starting: 5-shot test


✅ 5-shot done | Accuracy: 57.50% | Time: 123.22 s
💾 Result saved: /workspace/cosmos_cola_5shot_2.csv

🧹 [COSMOS] RAM and GPU cleared.

📄 Summary results saved → /workspace/llm_cosmos_results2_summary.csv


In [21]:
# Model loading updated: using `from_pretrained` fully fixes the `meta tensor` error

from tqdm import tqdm
import os
import time
import gc
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download

# Setup
shot_types = [0, 3, 5]
results = []

print(f"\n🎯 Starting! Selected models: {[name for _, name in models]}")
print(f"📚 Datasets: {list(datasets.keys())}")
print(f"🔁 Shot counts: {shot_types}")

# Model loop
for model_id, model_name in models:
    print(f"\n🚀 [{model_name.upper()}] downloading and loading model...")

    model_dir = f"/content/{model_name}_model"
    offload_dir = f"/content/offload_{model_name}"

    # Download model
    snapshot_download(
        repo_id=model_id,
        local_dir=model_dir,
        local_dir_use_symlinks=False
    )

    # Tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        device_map="auto",
        torch_dtype=torch.float16
    )

    print(f"✅ [{model_name.upper()}] model downloaded and loaded.")

    # Dataset loop
    for dataset_name, data in datasets.items():
        task_train = data["train"]
        all_examples = data["test"][:200]

        print(f"\n📂 Dataset: {dataset_name.upper()} ({len(all_examples)} examples)")

        for shot in shot_types:
            print(f"\n💡 Starting: {shot}-shot test")

            correct = 0
            total = 0
            predictions = []
            start_time = time.time()

            for example in tqdm(all_examples, desc=f"{model_name.upper()} | {dataset_name.upper()} | {shot}-shot"):
                prompt = get_few_shot_prompt(task_train, example["sentence"], shot=shot, task=dataset_name)
                try:
                    output = get_llm_prediction(prompt, model, tokenizer)
                    pred = parse_output(output, task=dataset_name)
                except Exception as e:
                    print(f"  ❌ Hata: {e}")
                    pred = -1

                gc.collect()
                torch.cuda.empty_cache()

                predictions.append({
                    "sentence": example["sentence"],
                    "label": example["label"],
                    "prediction": pred
                })

                if pred != -1:
                    total += 1
                    if pred == example["label"]:
                        correct += 1

            end_time = time.time()
            acc = correct / total if total > 0 else 0.0

            df_results = pd.DataFrame(predictions)
            df_results["correct"] = df_results["label"] == df_results["prediction"]
            df_results["model"] = model_name
            df_results["dataset"] = dataset_name
            df_results["shot"] = shot
            df_results["duration_sec"] = round(end_time - start_time, 2)

            # Save results
            out_path = f"/workspace/{model_name}_{dataset_name}_{shot}shot_2.csv"
            df_results.to_csv(out_path, index=False)

            print(f"✅ {shot}-shot done | Accuracy: {acc:.2%} | Time: {df_results['duration_sec'][0]} s")
            print(f"💾 Result saved: {out_path}")

            # Add summary to results
            results.append({
                "model": model_name,
                "dataset": dataset_name,
                "shot": shot,
                "accuracy": acc,
                "correct": correct,
                "total": total,
                "duration_sec": round(end_time - start_time, 2)
            })

    # Clean up after model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\n🧹 [{model_name.upper()}] RAM and GPU cleared.")

# Save summary output of all models
results_df = pd.DataFrame(results)
results_df.to_csv("/workspace/llm_turkcell_results2_summary.csv", index=False)
print("\n📄 Summary results saved → /workspace/llm_turkcell_results2_summary.csv")


🎯 Starting! Selected models: ['turkcell']
📚 Datasets: ['sst2', 'cola']
🔁 Shot counts: [0, 3, 5]

🚀 [TURKCELL] downloading and loading model...


✅ [TURKCELL] model downloaded and loaded.

📂 Dataset: SST2 (200 examples)

💡 Starting: 0-shot test


✅ 0-shot done | Accuracy: 0.00% | Time: 124.51 s
💾 Result saved: /workspace/turkcell_sst2_0shot_2.csv

💡 Starting: 3-shot test


✅ 3-shot done | Accuracy: 69.77% | Time: 124.92 s
💾 Result saved: /workspace/turkcell_sst2_3shot_2.csv

💡 Starting: 5-shot test


✅ 5-shot done | Accuracy: 55.41% | Time: 130.42 s
💾 Result saved: /workspace/turkcell_sst2_5shot_2.csv

📂 Dataset: COLA (200 examples)

💡 Starting: 0-shot test


✅ 0-shot done | Accuracy: 52.28% | Time: 121.79 s
💾 Result saved: /workspace/turkcell_cola_0shot_2.csv

💡 Starting: 3-shot test


✅ 3-shot done | Accuracy: 53.00% | Time: 122.43 s
💾 Result saved: /workspace/turkcell_cola_3shot_2.csv

💡 Starting: 5-shot test


✅ 5-shot done | Accuracy: 56.28% | Time: 123.78 s
💾 Result saved: /workspace/turkcell_cola_5shot_2.csv

🧹 [TURKCELL] RAM and GPU cleared.

📄 Summary results saved → /workspace/llm_turkcell_results2_summary.csv


In [25]:
# Model loading updated: using `from_pretrained` fully fixes the `meta tensor` error

from tqdm import tqdm
import os
import time
import gc
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download

# Setup
shot_types = [0, 3, 5]
results = []

print(f"\n🎯 Starting! Selected models: {[name for _, name in models]}")
print(f"📚 Datasets: {list(datasets.keys())}")
print(f"🔁 Shot counts: {shot_types}")

# Model loop
for model_id, model_name in models:
    print(f"\n🚀 [{model_name.upper()}] downloading and loading model...")

    model_dir = f"/content/{model_name}_model"
    offload_dir = f"/content/offload_{model_name}"

    # Download model
    snapshot_download(
        repo_id=model_id,
        local_dir=model_dir,
        local_dir_use_symlinks=False
    )

    # Tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        device_map="auto",
        torch_dtype=torch.float16
    )

    print(f"✅ [{model_name.upper()}] model downloaded and loaded.")

    # Dataset loop
    for dataset_name, data in datasets.items():
        task_train = data["train"]
        all_examples = data["test"][:200]

        print(f"\n📂 Dataset: {dataset_name.upper()} ({len(all_examples)} examples)")

        for shot in shot_types:
            print(f"\n💡 Starting: {shot}-shot test")

            correct = 0
            total = 0
            predictions = []
            start_time = time.time()

            for example in tqdm(all_examples, desc=f"{model_name.upper()} | {dataset_name.upper()} | {shot}-shot"):
                prompt = get_few_shot_prompt(task_train, example["sentence"], shot=shot, task=dataset_name)
                try:
                    output = get_llm_prediction(prompt, model, tokenizer)
                    pred = parse_output(output, task=dataset_name)
                except Exception as e:
                    print(f"  ❌ Hata: {e}")
                    pred = -1

                gc.collect()
                torch.cuda.empty_cache()

                predictions.append({
                    "sentence": example["sentence"],
                    "label": example["label"],
                    "prediction": pred
                })

                if pred != -1:
                    total += 1
                    if pred == example["label"]:
                        correct += 1

            end_time = time.time()
            acc = correct / total if total > 0 else 0.0

            df_results = pd.DataFrame(predictions)
            df_results["correct"] = df_results["label"] == df_results["prediction"]
            df_results["model"] = model_name
            df_results["dataset"] = dataset_name
            df_results["shot"] = shot
            df_results["duration_sec"] = round(end_time - start_time, 2)

            # Save results
            out_path = f"/workspace/{model_name}_{dataset_name}_{shot}shot_2.csv"
            df_results.to_csv(out_path, index=False)

            print(f"✅ {shot}-shot done | Accuracy: {acc:.2%} | Time: {df_results['duration_sec'][0]} s")
            print(f"💾 Result saved: {out_path}")

            # Add summary to results
            results.append({
                "model": model_name,
                "dataset": dataset_name,
                "shot": shot,
                "accuracy": acc,
                "correct": correct,
                "total": total,
                "duration_sec": round(end_time - start_time, 2)
            })

    # Clean up after model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\n🧹 [{model_name.upper()}] RAM and GPU cleared.")

# Save summary output of all models
results_df = pd.DataFrame(results)
results_df.to_csv("/workspace/llm_koc_results2_summary.csv", index=False)
print("\n📄 Summary results saved → /workspace/llm_koc_results2_summary.csv")


🎯 Starting! Selected models: ['koc']
📚 Datasets: ['sst2', 'cola']
🔁 Shot counts: [0, 3, 5]

🚀 [KOC] downloading and loading model...


✅ [KOC] model downloaded and loaded.

📂 Dataset: SST2 (200 examples)

💡 Starting: 0-shot test


❌ parse_output could not parse:
 Aşağıdaki cümlelerin duygu durumunu değerlendir. Cevabın sadece 'Olumlu' ya da 'Olumsuz' olsun.

Cümle: Anime tarzı bir film. Filmde gereksiz ilişkiler, uzun sahnelerle sıkıcı bir şekilde anlatılıyor. Bana hitap etmedi ve izlerken oldukça sıkıldım. Bu tarz filmlerden hoşlananlar için iyi bir film olabilir.
Cevap:


❌ parse_output could not parse:
 Aşağıdaki cümlelerin duygu durumunu değerlendir. Cevabın sadece 'Olumlu' ya da 'Olumsuz' olsun.

Cümle: Aman Tanrım dediğim ender filmlerden! Öyle feci gerildim ki mantık hatalarını göremedim :) Lütfen hamile olan bayanlar kazara merak edip izleyeyim demesin.. Şiddet, kan banyosu, delilik hepsi birbirini güzel tamamlamış. Mantık hatalarını fazla gözlersek eğer gerilemeyiz ki, böyle de bir gerçek var :) Filmi beğenenlerden oldum..
Cevap:


❌ parse_output could not parse:
 Aşağıdaki cümlelerin duygu durumunu değerlendir. Cevabın sadece 'Olumlu' ya da 'Olumsuz' olsun.

Cümle: Senaryosunu, Eric Jager'ın 29 Aralık 1386 tarihinde, bir Norman &şövalyesi (knight)& olan Jean de Carrouges ile eski dostu yaver (squire)& Jacques Le Gris arasındaki, Paris'te düzenlenen bir silahlı düello ile nihayetlenen 'GERÇEK' bir öyküyü anlattığı The Last Duel: A True Story of Trial by Combat in Medieval France den (2004) uyarlayarak Nicole Holofcener, Ben Affleck ve Matt Damon'ın yazdıkları The Last Duel&, yönetmen koltuğunda usta sinemacı Ridley Scott'ın oturduğu destansı bir tarihi drama olarak geliyor karşımıza... Gelin isterseniz; başroldeki üç ana karakterin, yaşanan olaylara yönelik farklı bakış açılarını yansıtan üç bölüm halinde kurgulanarak, 100 milyon dolarlık bir bütçeyle çekilen ve dünya prömiyeri 10 Eylül 2021 tarihinde Venedik Uluslararası Film Festivalinde yapılan filmimize biraz daha yakından bakalım... BÖLÜM BİR - Jean de Carro

...

❌ parse_output could not parse:
 Aşağıdaki cümlelerin gramer açısından doğru olup olmadığını değerlendir.

Cümle: Türkçenin yazımı için yetersiz olmasına rağmen Türklerce uzun süre kullanmışır.
Cevap: Hayır

Cümle: Eski Türklerde de, bugün olduğu gibi Vatan kutsaldı. Vatan Sevgisi, bir ahlaksızlık ölçüsüydü.
Cevap: Hayır

Cümle: Ses düşmesi, kelime türüne göre, tek ünlünün çöküşü veya bir veya birden fazla seslem çöküşü olarak uygulanmaktadır.
Cevap: Hayır

Cümle: Uygur alfabesi Sami kökenli Soğd alfabesinden, daha doğrusu bu alfabenin kürsiv, yani işlek türünden çıkmıştır.
Cevap: Evet

Cümle: Sözcük öbeklerinde, cümle ögelerinde yanlış yerlerde durgu veya durak yapıldığında anlaşılma güçleşebilir.
Cevap: Evet

Cümle: Katil beni merkezde sandılar.
Cevap: Evet
❌ parse_output could not parse:
 Aşağıdaki cümlelerin gramer açısından doğru olup olmadığını değerlendir.

Cümle: Öyleyse kültür nedir?
Cevap: Evet

Cümle: Yazı dili standardı bir kere belirlendikten sonra, eğitimlerini bu standar

❌ parse_output could not parse:
 Aşağıdaki cümlelerin gramer açısından doğru olup olmadığını değerlendir.

Cümle: Yazıda bilgisayar kullanmak yerleşmiş bir kuraldır.
Cevap: Evet

Cümle: Eski Türklerde de, bugün olduğu gibi Vatan kutsaldı. Vatan Sevgisi, bir ahlaksızlık ölçüsüydü.
Cevap: Hayır

Cümle: Bir akademik yazıda sadelik, akılcılık, neden-sonuç ilişkisi, sözü dolandırmadan ifade etme ve aranılan açıklık özelliklerdir.
Cevap: Hayır

Cümle: Günümüzde de Türkçede çok sayıda çeşitlenmeler vardır.
Cevap: Hayır

Cümle: Sahanın çok önemli adlarının vurgunun yeri, türü ve işlevleri konusunda çelişen tespitleri, bunu açıkça göstermektedir.
Cevap: Evet

Cümle: Bugün öğrendiğimiz konuyu akşam ayrıntılı olarak yüzeysel bir biçimde tekrarlayınız.
Cevap: Hayır


❌ parse_output could not parse:
 Aşağıdaki cümlelerin gramer açısından doğru olup olmadığını değerlendir.

Cümle: Dolaysıyla tip, bir toplumdaki insanların şahsında toplamış kişidir ortak özelliklerini.
Cevap: Hayır

Cümle: Türkçenin Yapısı ve İşletme Biçimi Yukarıda açıklandığı üzere Türkçe, eklemeli diller grubunun en tipik üyesidir.
Cevap: Evet

Cümle: Ürünlerin Türkiyede üretilmesine ve çoğunun da iç piyasada tüketilmesine karşılık **adların ve yabancı olması** düşündürücüdür.
Cevap: Hayır

Cümle: Dr. Abdurrahman Güzel, Türk halk Edebiyatı El Kitabı, 10. Baskı, Ankara 2017, s.319-320. 6 Bay: Zengin, varlıklı.
Cevap: Evet

Cümle: Ezgi dillerinde ezgi, sözcükler arasında değil, sözdizimsel yapılar arasında anlam farkı belirtilmede kullanır.
Cevap: Hayır

Cümle: Önemli rolü oynayan belli noktalar vardır seslerin oluşmasında söyleyiş organlarında.
Cevap: Evet


❌ parse_output could not parse:
 Aşağıdaki cümlelerin gramer açısından doğru olup olmadığını değerlendir.

Cümle: İlk kuşakta bu sistem daha çok pantomim özelliklerini taşıyordu ve konuşan insanların jestlerine benziyordu.
Cevap: Evet

Cümle: Her dilin kendine özgü bir yapı ve işleyişi vardır.
Cevap: Evet

Cümle: Diyalektler köken ve tarihsel gelişim bakımından en az ölçünlü diyalektler kadar eski ol-ayamaz.
Cevap: Hayır

Cümle: Karışık bir taş, demir ve cam yığını bir araya geldi mi, bir mimari yapı vücuda gelir.
Cevap: Hayır

Cümle: Ne babasının büyük bir saygı gösterdiği, kah görünüp kah çekilip giden birtakım adamlar?
Cevap: Hayır

Cümle: İsim-Fiil Grubu ve Sıfat-Fiil Grubu 72 neler öğrendik? 5 Seçenek hangisinde fiilimsiden türetilmiş tekrar grubuna yer verilmemiştir?
Cevap: Hayır


❌ parse_output could not parse:
 Aşağıdaki cümlelerin gramer açısından doğru olup olmadığını değerlendir.

Cümle: Bunların iki hece olarak yazılması az rastlanan bir durum değildir.
Cevap: Evet

Cümle: Her sahada, bilhassa medeniyet ālemine eser vermek için çalışkan olmayayı hedef tutmalıdır.
Cevap: Hayır

Cümle: Çeşitli tiyatro türleri vardır: a. Trajedi: Önemli ya da soylu kişilerin başlarına gelen kötü durumları konu alan oyunlardır.
Cevap: Evet

Cümle: Türkçe'de akıcı [r], vurmalı [ř] ve ötümsüz [r] sesleri bölütleyici dağılım içindedir.
Cevap: Hayır

Cümle: Papua Yeni Gine Fransa büyüklüğünde bir yerdir, ama değil.
Cevap: Hayır

Cümle: Dildeki değişme nedenlerinden bazılarına aşağıda değinilecektir.
Cevap: Evet
❌ parse_output could not parse:
 Aşağıdaki cümlelerin gramer açısından doğru olup olmadığını değerlendir.

Cümle: Ayın ve hemze hece sonunda düşerken söyleyişte kendisinden sonraki ünlüyü uzatabilir: meʾmur > mēmur, miʿmar > mīmar, teʾmin > tēmin.
Cevap: Evet

Cümle: Uzun v